# Clean the dataset

Runs the cleaning steps on the raw PhiUSIIL CSV, in order, and prints how many rows each step removes. Every step reads from the raw file, so running the notebook twice gives the same counts.

| Step | Rule |
|---|---|
| 1. Duplicate URLs | A row whose `URL` exactly matches another row's is a duplicate, and the first copy is kept. If copies of a URL have different labels, every copy is removed and counted separately. |

The steps live in `ml/cleaning/` and are tested in `ml/tests/`. This notebook only runs them. Deduplication happens here, before any train/test split.

Get the CSV first by following `ml/README.md`. Links in the data are only ever handled as text.

In [1]:
import sys
from pathlib import Path

import pandas as pd

CSV_NAME = "PhiUSIIL_Phishing_URL_Dataset.csv"


def find_ml_dir(start: Path) -> Path:
    """Walk up to the repo root (the folder holding AGENTS.md), so the notebook runs from any working directory inside the repo."""
    for folder in [start, *start.parents]:
        if (folder / "AGENTS.md").exists() and (folder / "ml").is_dir():
            return folder / "ml"
    raise FileNotFoundError("Could not find the ml/ folder. Open this notebook from inside the LinkGuard repo.")


ML_DIR = find_ml_dir(Path.cwd().resolve())
DATA_PATH = ML_DIR / "data" / CSV_NAME
sys.path.insert(0, str(ML_DIR))

from cleaning.duplicates import remove_duplicate_urls  # noqa: E402

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}.\n"
        "Download it by following 'Get the dataset' in ml/README.md, then run this notebook again."
    )

raw = pd.read_csv(DATA_PATH)
print(f"Raw file: {len(raw):,} rows")

Raw file: 235,795 rows


## Step 1: duplicate URLs

In [2]:
duplicates = remove_duplicate_urls(raw)

print(f"Rows before:                    {duplicates.rows_before:,}")
print(f"Extra copies removed:           {duplicates.duplicate_rows_removed:,}")
print(f"URLs with both labels:          {duplicates.conflicting_urls:,}")
print(f"  rows removed for that reason: {duplicates.conflicting_rows_removed:,}")
print(f"Rows after:                     {duplicates.rows_after:,}")

Rows before:                    235,795
Extra copies removed:           425
URLs with both labels:          0
  rows removed for that reason: 0
Rows after:                     235,370


## Why this is not 808

Other analyses report about 808 duplicates. The difference comes from what counts as a duplicate:

- **808** is the number of rows whose **numeric feature values** exactly match an earlier row's, even though their URLs are different.
- **This step** follows the story's spec: a duplicate is a row with the **same `URL`**.

The cell below works out both numbers from the raw file.

In [3]:
same_url = raw.duplicated("URL").sum()
same_numeric_features = raw.select_dtypes("number").duplicated().sum()

print(f"Extra rows with the same URL:                 {same_url:,}")
print(f"Extra rows with the same numeric features:    {same_numeric_features:,}")

Extra rows with the same URL:                 425
Extra rows with the same numeric features:    808


## What the duplicates look like

The cell below shows how many copies each duplicated URL has, which label they carry, and how often the two copies' features disagree.

In [4]:
copies = raw[raw.duplicated("URL", keep=False)]
copies_per_url = copies.groupby("URL").size()

print(f"Duplicated URLs: {copies_per_url.size:,}")
print(f"Copies per duplicated URL: {copies_per_url.value_counts().sort_index().to_dict()}  (copies: number of URLs)")
print(f"Raw label of their copies: {copies['label'].value_counts().to_dict()}  (raw file: 0 = phishing)")

feature_columns = [column for column in raw.columns if column not in ("FILENAME", "URL", "label")]
differs = copies.groupby("URL")[feature_columns].nunique().gt(1)
print(f"URLs whose two copies differ in at least one feature: {differs.any(axis=1).sum():,} of {copies_per_url.size:,}")
print(f"URLs whose copies even differ in URLLength:           {differs['URLLength'].sum():,}")

Duplicated URLs: 425
Copies per duplicated URL: {2: 425}  (copies: number of URLs)
Raw label of their copies: {0: 850}  (raw file: 0 = phishing)
URLs whose two copies differ in at least one feature: 425 of 425
URLs whose copies even differ in URLLength:           425


Every duplicated URL appears exactly twice, and both copies are labelled phishing. The two copies are never identical rows. Every pair differs in its page features, and even in `URLLength`, although the URL text is exactly the same. So the feature values weren't worked out from the URL text alone, and each copy is probably a separate crawl of the same link.

The spec's rule is to keep the first copy. Both copies share a label, so the choice doesn't change the label balance. It does mean the kept row's features come from one crawl only.

## Row counts by step

Later cleaning steps add a row to this table.

In [5]:
steps = pd.DataFrame(
    [
        {"Step": "Raw file", "Rows removed": 0, "Rows left": len(raw)},
        {
            "Step": "1. Duplicate URLs",
            "Rows removed": duplicates.rows_before - duplicates.rows_after,
            "Rows left": duplicates.rows_after,
        },
    ]
)
print(steps.to_string(index=False))

             Step  Rows removed  Rows left
         Raw file             0     235795
1. Duplicate URLs           425     235370
